# Week 79, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w079d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('miniflask.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of Flask that runs in the page (Week 77).\n\nThe names and behaviour follow Flask\'s own, so code written against this module also runs with\nthe real Flask (pip install flask): Flask, request, g, jsonify, abort, redirect, make_response,\nurl_for, Blueprint, and app.test_client(). It\'s a teaching copy, not the real thing: no templates,\nsessions, streaming or debugger.\n"""\nimport io\nimport json\nimport re\nfrom contextvars import ContextVar\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qs, urlencode, urlsplit\n\n_current = ContextVar("request")\n\n\nclass HTTPException(Exception):\n    def __init__(self, code, description=None):\n        self.code = code\n        self.name = HTTPStatus(code).phrase\n        self.description = description or self.name\n        super().__init__(f"{code} {self.name}: {self.description}")\n\n\ndef abort(code, description=None):\n    raise HTTPException(code, description)\n\n\nclass Headers:\n    """Case-insensitive headers."""\n    def __init__(self, items=()):\n        self._items = [(k, str(v)) for k, v in (items.items() if isinstance(items, dict) else items)]\n\n    def get(self, name, default=None):\n        for k, v in self._items:\n            if k.lower() == name.lower():\n                return v\n        return default\n\n    def __getitem__(self, name):\n        value = self.get(name)\n        if value is None:\n            raise KeyError(name)\n        return value\n\n    def __contains__(self, name):\n        return self.get(name) is not None\n\n    def __setitem__(self, name, value):\n        self._items = [(k, v) for k, v in self._items if k.lower() != name.lower()] + [(name, str(value))]\n\n    def items(self):\n        return list(self._items)\n\n\nclass Args(dict):\n    """Query or form values: .get(name) gives the first value; .getlist(name) all of them."""\n    def __init__(self, pairs):\n        super().__init__()\n        self._lists = {}\n        for k, v in pairs:\n            self._lists.setdefault(k, []).append(v)\n            dict.setdefault(self, k, v)\n\n    def get(self, key, default=None, type=None):\n        if key not in self:\n            return default\n        value = self[key]\n        if type is not None:\n            try:\n                return type(value)\n            except (TypeError, ValueError):\n                return default\n        return value\n\n    def getlist(self, key):\n        return list(self._lists.get(key, []))\n\n\nclass Request:\n    def __init__(self, environ):\n        self.environ = environ\n        self.method = environ["REQUEST_METHOD"].upper()\n        self.path = environ.get("PATH_INFO", "/") or "/"\n        self.args = Args(_pairs(environ.get("QUERY_STRING", "")))\n        items = [(k[5:].replace("_", "-").title(), v) for k, v in environ.items() if k.startswith("HTTP_")]\n        if environ.get("CONTENT_TYPE"):\n            items.append(("Content-Type", environ["CONTENT_TYPE"]))\n        self.headers = Headers(items)\n        length = int(environ.get("CONTENT_LENGTH") or 0)\n        self.data = environ["wsgi.input"].read(length) if length else b""\n        self.content_type = environ.get("CONTENT_TYPE", "")\n        self.form = Args(_pairs(self.data.decode("utf-8"))) if self.content_type.startswith("application/x-www-form-urlencoded") else Args([])\n        self.view_args = {}\n        cookie = environ.get("HTTP_COOKIE", "")\n        self.cookies = dict(p.strip().split("=", 1) for p in cookie.split(";") if "=" in p)\n\n    @property\n    def is_json(self):\n        return self.content_type.split(";")[0].strip() == "application/json"\n\n    def get_json(self, silent=False, force=False):\n        if not (self.is_json or force):\n            if silent:\n                return None\n            abort(415, "expected a JSON body (Content-Type: application/json)")\n        try:\n            return json.loads(self.data or b"null")\n        except ValueError:\n            if silent:\n                return None\n            abort(400, "the body isn\'t valid JSON")\n\n    @property\n    def json(self):\n        return self.get_json()\n\n\ndef _pairs(text):\n    return [(k, v) for k, vs in parse_qs(text, keep_blank_values=True).items() for v in vs]\n\n\nclass _Proxy:\n    def __init__(self, getter):\n        object.__setattr__(self, "_getter", getter)\n\n    def __getattr__(self, name):\n        return getattr(self._getter(), name)\n\n    def __setattr__(self, name, value):\n        setattr(self._getter(), name, value)\n\n\ndef _request():\n    try:\n        return _current.get()[0]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\ndef _g():\n    try:\n        return _current.get()[1]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\nclass _G:\n    def get(self, name, default=None):\n        return self.__dict__.get(name, default)\n\n\nrequest = _Proxy(_request)\ng = _Proxy(_g)\n\n\nclass Response:\n    def __init__(self, response=b"", status=200, headers=None, mimetype=None, content_type=None):\n        if isinstance(response, str):\n            response = response.encode("utf-8")\n        self.data = response\n        self.status_code = status\n        self.headers = Headers(headers or {})\n        if content_type or mimetype or "Content-Type" not in self.headers:\n            ctype = content_type or (f"{mimetype}; charset=utf-8" if mimetype and mimetype.startswith("text/") else mimetype) or "text/html; charset=utf-8"\n            self.headers["Content-Type"] = ctype\n\n    @property\n    def status(self):\n        return f"{self.status_code} {HTTPStatus(self.status_code).phrase}"\n\n    @property\n    def text(self):\n        return self.data.decode("utf-8")\n\n    def get_json(self):\n        return json.loads(self.data)\n\n    @property\n    def json(self):\n        return self.get_json()\n\n    def set_cookie(self, key, value="", max_age=None, path="/", secure=False, httponly=False, samesite=None):\n        parts = [f"{key}={value}"]\n        if max_age is not None:\n            parts.append(f"Max-Age={max_age}")\n        parts.append(f"Path={path}")\n        if secure:\n            parts.append("Secure")\n        if httponly:\n            parts.append("HttpOnly")\n        if samesite:\n            parts.append(f"SameSite={samesite}")\n        self.headers._items.append(("Set-Cookie", "; ".join(parts)))\n\n\ndef jsonify(*args, **kwargs):\n    data = args[0] if len(args) == 1 else (list(args) if args else kwargs)\n    return Response(json.dumps(data), headers={"Content-Type": "application/json"})\n\n\ndef redirect(location, code=302):\n    return Response(b"", code, {"Location": location})\n\n\ndef make_response(*args):\n    rv = args[0] if len(args) == 1 else args\n    return _App._to_response(rv)\n\n\ndef url_for(endpoint, **values):\n    return _current.get()[2].url_for(endpoint, **values)\n\n\n_CONVERTERS = {"int": (r"\\d+", int), "float": (r"\\d+(?:\\.\\d+)?", float), "path": (r".+", str), "string": (r"[^/]+", str)}\n\n\nclass _Rule:\n    def __init__(self, rule, endpoint, view, methods):\n        self.rule, self.endpoint, self.view = rule, endpoint, view\n        self.methods = {m.upper() for m in methods} | ({"HEAD"} if "GET" in {m.upper() for m in methods} else set())\n        regex, self.converters = "", {}\n        for piece in re.split(r"(<[^>]+>)", rule):\n            m = re.fullmatch(r"<(?:(\\w+):)?(\\w+)>", piece)\n            if m:\n                kind, name = m.group(1) or "string", m.group(2)\n                pattern, convert = _CONVERTERS[kind]\n                regex += f"(?P<{name}>{pattern})"\n                self.converters[name] = convert\n            else:\n                regex += re.escape(piece)\n        self.regex = re.compile(regex)\n\n    def match(self, path):\n        m = self.regex.fullmatch(path)\n        if not m:\n            return None\n        return {k: self.converters[k](v) for k, v in m.groupdict().items()}\n\n    def build(self, values):\n        def fill(m):\n            return str(values.pop(m.group(2)))\n        path = re.sub(r"<(?:(\\w+):)?(\\w+)>", fill, self.rule)\n        return path + (f"?{urlencode(values)}" if values else "")\n\n\nclass _Setup:\n    def route(self, rule, methods=("GET",), endpoint=None):\n        def decorate(view):\n            self._add(rule, endpoint or view.__name__, view, methods)\n            return view\n        return decorate\n\n    def get(self, rule, **kw):\n        return self.route(rule, methods=["GET"], **kw)\n\n    def post(self, rule, **kw):\n        return self.route(rule, methods=["POST"], **kw)\n\n    def put(self, rule, **kw):\n        return self.route(rule, methods=["PUT"], **kw)\n\n    def patch(self, rule, **kw):\n        return self.route(rule, methods=["PATCH"], **kw)\n\n    def delete(self, rule, **kw):\n        return self.route(rule, methods=["DELETE"], **kw)\n\n\nclass Blueprint(_Setup):\n    def __init__(self, name, import_name, url_prefix=""):\n        self.name, self.url_prefix = name, url_prefix\n        self.deferred = []\n\n    def _add(self, rule, endpoint, view, methods):\n        self.deferred.append((rule, endpoint, view, methods))\n\n\nclass _App(_Setup):\n    def __init__(self, import_name):\n        self.import_name = import_name\n        self.config = {"TESTING": False}\n        self.rules = []\n        self.before_funcs, self.after_funcs, self.error_handlers = [], [], {}\n\n    def _add(self, rule, endpoint, view, methods):\n        self.rules.append(_Rule(rule, endpoint, view, methods))\n\n    def register_blueprint(self, bp, url_prefix=None):\n        prefix = url_prefix if url_prefix is not None else bp.url_prefix\n        for rule, endpoint, view, methods in bp.deferred:\n            self._add(prefix + rule, f"{bp.name}.{endpoint}", view, methods)\n\n    def before_request(self, f):\n        self.before_funcs.append(f)\n        return f\n\n    def after_request(self, f):\n        self.after_funcs.append(f)\n        return f\n\n    def errorhandler(self, code):\n        def decorate(f):\n            self.error_handlers[code] = f\n            return f\n        return decorate\n\n    def url_for(self, endpoint, **values):\n        for r in self.rules:\n            if r.endpoint == endpoint:\n                return r.build(dict(values))\n        raise LookupError(f"no endpoint {endpoint!r}")\n\n    @staticmethod\n    def _to_response(rv):\n        status, headers = None, None\n        if isinstance(rv, tuple):\n            if len(rv) == 3:\n                rv, status, headers = rv\n            elif isinstance(rv[1], (dict, list)):\n                rv, headers = rv\n            else:\n                rv, status = rv\n        if isinstance(rv, Response):\n            response = rv\n        elif isinstance(rv, (dict, list)):\n            response = jsonify(rv)\n        elif isinstance(rv, (str, bytes)):\n            response = Response(rv)\n        elif rv is None:\n            raise TypeError("the view function did not return a valid response")\n        else:\n            raise TypeError(f"can\'t turn {type(rv).__name__} into a response")\n        if status is not None:\n            response.status_code = status\n        for k, v in (headers.items() if isinstance(headers, dict) else headers or []):\n            response.headers[k] = v\n        return response\n\n    def _error(self, e):\n        handler = self.error_handlers.get(e.code)\n        if handler:\n            return self._to_response(handler(e))\n        body = f"<h1>{e.code} {e.name}</h1><p>{e.description}</p>"\n        return Response(body, e.code)\n\n    def _dispatch(self, req):\n        path = req.path\n        allowed = set()\n        for r in self.rules:\n            values = r.match(path)\n            if values is None:\n                continue\n            if req.method in r.methods:\n                req.view_args = values\n                return r.view(**values)\n            allowed |= r.methods\n        if allowed:\n            raise _MethodNotAllowed(sorted(allowed | {"OPTIONS"}))\n        abort(404, "The requested URL was not found on the server.")\n\n    def full_dispatch(self, environ):\n        req = Request(environ)\n        token = _current.set((req, _G(), self))\n        try:\n            try:\n                rv = None\n                for f in self.before_funcs:\n                    rv = f()\n                    if rv is not None:\n                        break\n                if rv is None:\n                    rv = self._dispatch(req)\n                response = self._to_response(rv)\n            except _MethodNotAllowed as e:\n                response = self._error(HTTPException(405))\n                response.headers["Allow"] = ", ".join(e.allowed)\n            except HTTPException as e:\n                response = self._error(e)\n            except Exception as e:\n                if self.config.get("TESTING") and 500 not in self.error_handlers:\n                    raise\n                response = self._error(HTTPException(500)) if 500 not in self.error_handlers else self._to_response(self.error_handlers[500](e))\n                response.status_code = 500\n            for f in self.after_funcs:\n                response = f(response) or response\n            if req.method == "HEAD":\n                response.headers["Content-Length"] = len(response.data)\n                response.data = b""\n            return response\n        finally:\n            _current.reset(token)\n\n    def __call__(self, environ, start_response):\n        response = self.full_dispatch(environ)\n        if "Content-Length" not in response.headers:\n            response.headers["Content-Length"] = len(response.data)\n        start_response(response.status, response.headers.items())\n        return [response.data]\n\n    def test_client(self):\n        return _TestClient(self)\n\n\nclass _MethodNotAllowed(Exception):\n    def __init__(self, allowed):\n        self.allowed = allowed\n\n\nclass Flask(_App):\n    pass\n\n\nclass _TestClient:\n    def __init__(self, app):\n        self.app = app\n        self.cookies = {}\n\n    def open(self, path, method="GET", json=None, data=None, headers=None, query_string=None):\n        url = urlsplit(path)\n        query = url.query\n        if query_string:\n            query = (query + "&" if query else "") + urlencode(query_string, doseq=True)\n        body, ctype = b"", None\n        if json is not None:\n            import json as _json\n            body, ctype = _json.dumps(json).encode("utf-8"), "application/json"\n        elif isinstance(data, dict):\n            body, ctype = urlencode(data, doseq=True).encode("utf-8"), "application/x-www-form-urlencoded"\n        elif data is not None:\n            body = data.encode("utf-8") if isinstance(data, str) else data\n        environ = {"REQUEST_METHOD": method.upper(), "PATH_INFO": url.path or "/", "QUERY_STRING": query,\n                   "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.url_scheme": "http",\n                   "wsgi.input": io.BytesIO(body), "CONTENT_LENGTH": str(len(body))}\n        headers = dict(headers or {})\n        if ctype and not any(k.lower() == "content-type" for k in headers):\n            environ["CONTENT_TYPE"] = ctype\n        for k, v in headers.items():\n            if k.lower() == "content-type":\n                environ["CONTENT_TYPE"] = v\n            else:\n                environ["HTTP_" + k.upper().replace("-", "_")] = v\n        if self.cookies:\n            environ["HTTP_COOKIE"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())\n        response = self.app.full_dispatch(environ)\n        for k, v in response.headers.items():\n            if k.lower() == "set-cookie":\n                name, _, rest = v.partition("=")\n                self.cookies[name] = rest.split(";")[0]\n        return response\n\n    def get(self, path, **kw):\n        return self.open(path, "GET", **kw)\n\n    def post(self, path, **kw):\n        return self.open(path, "POST", **kw)\n\n    def put(self, path, **kw):\n        return self.open(path, "PUT", **kw)\n\n    def patch(self, path, **kw):\n        return self.open(path, "PATCH", **kw)\n\n    def delete(self, path, **kw):\n        return self.open(path, "DELETE", **kw)\n')

**Predict before you run:** what will this print?



In [ ]:
import html
name = '<b>Ada</b> & "friends"'
print(html.escape(name))
print(html.escape(name, quote=False))

**Predict before you run:** what will this print?



In [ ]:
from pathlib import Path
base = Path("/srv/uploads/7")
for name in ["notes.txt", "../8/notes.txt", "/etc/passwd"]:
    print((base / name).resolve().is_relative_to(base))

**Predict before you run:** what will this print?



In [ ]:
env = {"DEBUG": "False", "WORKERS": "4"}
print(bool(env["DEBUG"]), env["DEBUG"].lower() in ("1", "true"), bool(env.get("MISSING")))

**Predict before you run:** what will this print?



In [ ]:
import hashlib, hmac
a = hmac.new(b"key", b"user=7", hashlib.sha256).hexdigest()
b = hmac.new(b"key", b"user=8", hashlib.sha256).hexdigest()
print(len(a), a == b, hmac.compare_digest(a, a))

**Predict before you run:** what will this print?



In [ ]:
email = "' OR 1=1 --"
print(f"SELECT id FROM users WHERE email = '{email}' AND password = 'x'")

### Exercise: Expiring download links

Write `make_link(path, expires, key)`: `PATH?expires=EXPIRES&sig=SIG`, where SIG is the hex HMAC-SHA256 (with the bytes `key`) of the text `PATH|EXPIRES`. Then `check_link(url, now, key)`: split the URL into path and query (`urllib.parse.urlsplit`, `parse_qs`), and return `"bad signature"` if `expires` or `sig` is missing, `expires` isn't a whole number, or the signature doesn't match (checked with `hmac.compare_digest`); `"expired"` if `now` is after `expires`; otherwise `"ok"`. Check the signature **before** the time, so a forged link never gets a different answer from a real one.

In [ ]:
import hashlib
import hmac
from urllib.parse import parse_qs, urlsplit

def make_link(path, expires, key):
    ___

def check_link(url, now, key):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_links():\n    \"signed and expiring\"\n    key = b\"link key\"\n    link = make_link(\"/files/report.pdf\", 1800000000, key)\n    assert link.startswith(\"/files/report.pdf?expires=1800000000&sig=\") and len(link.rsplit(\"=\", 1)[1]) == 64\n    assert check_link(link, 1799999999, key) == \"ok\" and check_link(link, 1800000000, key) == \"ok\"\n    assert check_link(link, 1800000001, key) == \"expired\"\n    forged = link.replace(\"1800000000\", \"1900000000\")\n    assert check_link(forged, 1850000000, key) == \"bad signature\" and check_link(forged, 1950000000, key) == \"bad signature\"\n    assert check_link(link.replace(\"report\", \"salaries\"), 0, key) == \"bad signature\"\n    for bad in (\"/files/x.pdf\", \"/files/x.pdf?expires=soon&sig=00\", link, ):\n        assert check_link(bad, 0, b\"other key\" if bad == link else key) == \"bad signature\"")

### Exercise: A safe report table

Write `report_table(rows, columns)`: an HTML table of `rows` (dicts) showing the requested `columns`, which must all be in the allow-list `ALLOWED` (raise `ValueError(f"unknown column: {name!r}")` otherwise, before building anything). The result is `<table>`, a header row `<tr><th>NAME</th>...</tr>`, one `<tr><td>VALUE</td>...</tr>` per row (missing values as empty cells; every value converted with `str` and escaped with `html.escape`), then `</table>`, all joined with no separators.

In [ ]:
import html

ALLOWED = ["name", "email", "plan", "joined"]

def report_table(rows, columns):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_table():\n    \"escaped cells, allowed columns\"\n    rows = [{\"name\": \"Ada <script>\", \"email\": \"a@x.org\", \"plan\": \"pro & more\", \"password\": \"hunter2\"}, {\"name\": \"Bob\", \"joined\": 2026}]\n    got = report_table(rows, [\"name\", \"plan\", \"joined\"])\n    assert got == (\"<table><tr><th>name</th><th>plan</th><th>joined</th></tr>\"\n                   \"<tr><td>Ada &lt;script&gt;</td><td>pro &amp; more</td><td></td></tr>\"\n                   \"<tr><td>Bob</td><td></td><td>2026</td></tr></table>\")\n    for cols in ([\"name\", \"password\"], [\"name\", \"<b>x</b>\"]):\n        try:\n            report_table(rows, cols)\n        except ValueError as e:\n            assert str(e) == f\"unknown column: {cols[1]!r}\"\n        else:\n            raise AssertionError(cols)")

### Exercise: Forge: a security review

Fix `create_app` in the starter as described above. Keep the routes, the JSON shapes and the helper functions.

In [ ]:
import base64
import hashlib
import hmac
import html
import secrets
import sqlite3
from pathlib import Path

from miniflask import Flask, abort, jsonify, make_response, request

ITERATIONS = 1000   # far too few for real use (around 600,000 is the guidance); kept low so the checks run quickly


def pbkdf2_sha256(password, salt, iterations, dklen=32):
    base = hmac.new(password, digestmod=hashlib.sha256)
    out, block = b"", 1
    while len(out) < dklen:
        h = base.copy()
        h.update(salt + block.to_bytes(4, "big"))
        u = h.digest()
        total = int.from_bytes(u, "big")
        for _ in range(iterations - 1):
            h = base.copy()
            h.update(u)
            u = h.digest()
            total ^= int.from_bytes(u, "big")
        out += total.to_bytes(32, "big")
        block += 1
    return out[:dklen]


def hash_password(password):
    salt = secrets.token_bytes(16)
    digest = pbkdf2_sha256(password.encode(), salt, ITERATIONS)
    return f"pbkdf2_sha256${ITERATIONS}${base64.b64encode(salt).decode()}${base64.b64encode(digest).decode()}"


def check_password(password, stored):
    try:
        algorithm, iterations, salt, digest = stored.split("$")
        expected = pbkdf2_sha256(password.encode(), base64.b64decode(salt), int(iterations))
        return algorithm == "pbkdf2_sha256" and hmac.compare_digest(expected, base64.b64decode(digest))
    except ValueError:
        return False


SCHEMA = """
CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY, email TEXT UNIQUE NOT NULL, password TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS notes (id INTEGER PRIMARY KEY, owner_id INTEGER NOT NULL, title TEXT NOT NULL, body TEXT NOT NULL);
"""


def create_app(env, conn, upload_dir, log=print):
    app = Flask(__name__)
    SECRET_KEY = env.get("SECRET_KEY", "dev")
    DEBUG = bool(env.get("DEBUG", False))
    conn.executescript(SCHEMA)

    def current_user():
        session = request.cookies.get("session")
        if not session:
            abort(401, "log in first")
        return int(session)

    @app.post("/signup")
    def signup():
        data = request.get_json()
        pw = hashlib.sha256(data["password"].encode()).hexdigest()
        cur = conn.execute(f"INSERT INTO users (email, password) VALUES ('{data['email']}', '{pw}')")
        return {"id": cur.lastrowid}, 201

    @app.post("/login")
    def login():
        data = request.get_json()
        log(f"login attempt: {data}")
        pw = hashlib.sha256(data["password"].encode()).hexdigest()
        row = conn.execute(f"SELECT id FROM users WHERE email = '{data['email']}' AND password = '{pw}'").fetchone()
        if row is None:
            exists = conn.execute(f"SELECT 1 FROM users WHERE email = '{data['email']}'").fetchone()
            return {"error": "wrong password" if exists else "no such user"}, 401
        response = make_response(jsonify({"ok": True}))
        response.set_cookie("session", str(row[0]))
        return response

    @app.get("/notes")
    def list_notes():
        user = current_user()
        q = request.args.get("q", "")
        rows = conn.execute(f"SELECT id, title FROM notes WHERE owner_id = {user} AND title LIKE '%{q}%' ORDER BY id").fetchall()
        return {"notes": [{"id": i, "title": t} for i, t in rows]}

    @app.post("/notes")
    def add_note():
        user = current_user()
        data = request.get_json()
        cur = conn.execute("INSERT INTO notes (owner_id, title, body) VALUES (?, ?, ?)", (user, data["title"], data["body"]))
        return {"id": cur.lastrowid}, 201

    @app.get("/notes/<int:note_id>")
    def get_note(note_id):
        current_user()
        row = conn.execute("SELECT id, title, body FROM notes WHERE id = ?", (note_id,)).fetchone()
        if row is None:
            abort(404)
        return {"id": row[0], "title": row[1], "body": row[2]}

    @app.get("/notes/<int:note_id>/page")
    def note_page(note_id):
        note = get_note(note_id)
        return f"<h1>{note['title']}</h1><div>{note['body']}</div>"

    @app.get("/files")
    def get_file():
        user = current_user()
        name = request.args.get("name", "")
        return Path(upload_dir, str(user), name).read_bytes()

    @app.errorhandler(500)
    def server_error(e):
        return {"error": f"{type(e).__name__}: {e}"}, 500

    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("KEY = \"k\" * 40\n\ndef _app(env=None, logs=None):\n    import shutil, sqlite3\n    from pathlib import Path\n    shutil.rmtree(\"uploads\", ignore_errors=True)\n    Path(\"uploads/1\").mkdir(parents=True)\n    Path(\"uploads/2\").mkdir(parents=True)\n    Path(\"uploads/1/plan.txt\").write_text(\"ada's plan\")\n    Path(\"uploads/2/diary.txt\").write_text(\"bob's diary\")\n    Path(\"server.key\").write_text(\"TOP SECRET\")\n    conn = sqlite3.connect(\":memory:\")\n    app = create_app({\"SECRET_KEY\": KEY} if env is None else env, conn, \"uploads\", (logs if logs is not None else []).append)\n    return app, conn\n\ndef _client(app, email, password):\n    c = app.test_client()\n    c.post(\"/signup\", json={\"email\": email, \"password\": password})\n    r = c.post(\"/login\", json={\"email\": email, \"password\": password})\n    assert r.status_code == 200, \"signing up and logging in should work\"\n    return c\n\ndef test_working_app():\n    \"the app still does its job\"\n    app, conn = _app()\n    ada = _client(app, \"ada@example.org\", \"correct horse battery\")\n    assert ada.post(\"/notes\", json={\"title\": \"Plans\", \"body\": \"Build an engine\"}).status_code == 201\n    assert ada.get(\"/notes\").get_json() == {\"notes\": [{\"id\": 1, \"title\": \"Plans\"}]}\n    assert ada.get(\"/notes/1\").get_json() == {\"id\": 1, \"title\": \"Plans\", \"body\": \"Build an engine\"}\n    assert ada.get(\"/notes?q=pla\").get_json()[\"notes\"][0][\"title\"] == \"Plans\"\n    assert ada.get(\"/files?name=plan.txt\").data == b\"ada's plan\"\n    assert app.test_client().get(\"/notes\").status_code == 401\n\ndef test_settings():\n    \"no secret, no start; DEBUG must really mean true\"\n    for env in ({}, {\"SECRET_KEY\": \"dev\"}):\n        try:\n            _app(env)\n        except RuntimeError as e:\n            assert str(e) == \"SECRET_KEY must be set (at least 32 characters)\"\n        else:\n            raise AssertionError(f\"started with {env}\")\n\ndef test_passwords_and_login():\n    \"salted slow hashes, one error message, no injection, nothing secret in the logs\"\n    logs = []\n    app, conn = _app(logs=logs)\n    _client(app, \"ada@example.org\", \"same password 1\")\n    _client(app, \"bob@example.org\", \"same password 1\")\n    stored = [r[0] for r in conn.execute(\"SELECT password FROM users ORDER BY id\")]\n    assert stored[0] != stored[1] and stored[0].startswith(\"pbkdf2_sha256$\"), \"use hash_password\"\n    c = app.test_client()\n    wrong_user = c.post(\"/login\", json={\"email\": \"nobody@example.org\", \"password\": \"x\"})\n    wrong_pw = c.post(\"/login\", json={\"email\": \"ada@example.org\", \"password\": \"x\"})\n    assert wrong_user.status_code == wrong_pw.status_code == 401 and wrong_user.get_json() == wrong_pw.get_json() == {\"error\": \"wrong email or password\"}\n    for email in (\"' OR 1=1 --\", \"ada@example.org' --\"):\n        assert c.post(\"/login\", json={\"email\": email, \"password\": \"x\"}).status_code == 401, email\n    assert not any(\"same password 1\" in line for line in logs), \"never log passwords\"\n    assert c.post(\"/signup\", json={\"email\": \"x', 'y'); DROP TABLE users; --\", \"password\": \"p\"}).status_code == 201\n    assert conn.execute(\"SELECT count(*) FROM users\").fetchone()[0] == 3\n\ndef test_sessions_and_access():\n    \"signed sessions, and only your own notes and files\"\n    app, conn = _app()\n    ada = _client(app, \"ada@example.org\", \"pw-ada-123\")\n    bob = _client(app, \"bob@example.org\", \"pw-bob-123\")\n    ada.post(\"/notes\", json={\"title\": \"Ada private\", \"body\": \"secret plans\"})\n    bob.post(\"/notes\", json={\"title\": \"Bob note\", \"body\": \"hello\"})\n    assert \".\" in ada.cookies[\"session\"], \"sign the session cookie\"\n    forger = app.test_client()\n    for forged in (\"1\", \"1.abc\", ada.cookies[\"session\"].replace(\"1.\", \"2.\", 1)):\n        forger.cookies[\"session\"] = forged\n        assert forger.get(\"/notes\").status_code == 401, forged\n    assert bob.get(\"/notes/1\").status_code == 404 and bob.get(\"/notes/1/page\").status_code == 404\n    assert bob.get(\"/notes\", query_string={\"q\": \"' OR '1'='1\"}).get_json() == {\"notes\": []}\n    assert bob.get(\"/notes\", query_string={\"q\": \"%\"}).get_json() == {\"notes\": []}\n    for name in (\"../2/diary.txt\", \"../../server.key\", \"/etc/passwd\", \"\"):\n        r = ada.get(\"/files\", query_string={\"name\": name})\n        assert r.status_code in (403, 404) and b\"diary\" not in r.data and b\"SECRET\" not in r.data, name\n    assert ada.get(\"/files\", query_string={\"name\": \"missing.txt\"}).status_code == 404\n\ndef test_output_and_errors():\n    \"escaped pages and errors that don't leak\"\n    logs = []\n    app, conn = _app(logs=logs)\n    ada = _client(app, \"ada@example.org\", \"pw-ada-123\")\n    ada.post(\"/notes\", json={\"title\": \"<script>steal()</script>\", \"body\": \"<img src=x onerror=alert(1)>\"})\n    page = ada.get(\"/notes/1/page\").data.decode()\n    assert \"<script>\" not in page and \"&lt;script&gt;\" in page and \"<img\" not in page\n    r = ada.post(\"/notes\", json={\"title\": \"no body\"})\n    assert r.status_code == 500 and set(r.get_json()) == {\"error\", \"incident\"} and \"body\" not in str(r.get_json())\n    assert any(r.get_json()[\"incident\"] in line for line in logs), \"log the details with the incident id\"\n    app2, _ = _app({\"SECRET_KEY\": KEY, \"DEBUG\": \"False\"})\n    c = _client(app2, \"a@example.org\", \"pw-123456\")\n    assert \"detail\" not in c.post(\"/notes\", json={\"title\": \"x\"}).get_json(), \"DEBUG=False must mean off\"\n    app3, _ = _app({\"SECRET_KEY\": KEY, \"DEBUG\": \"true\"})\n    c = _client(app3, \"a@example.org\", \"pw-123456\")\n    assert \"detail\" in c.post(\"/notes\", json={\"title\": \"x\"}).get_json()")

> **How the holes fitted together.** Look at the original code again and notice how the problems combined. The SQL injection in login bypassed the password check, and the unsigned cookie meant nobody needed to log in at all; either one alone gave away every account. The missing ownership check would have survived both fixes: broken access control is a logic bug that no library prevents, so it needs tests like the ones here, written from an attacker's point of view. Which of the fixes would a tool such as Bandit have found on its own (the f-string SQL, the weak hash), and which needed a person reading the code with the week's questions in mind?

### Exercise: Forge bonus: an automatic reviewer

Write `review(source)`: parse the code with `ast.parse` and return a sorted list of `(line, rule)` pairs (no duplicates) for: `"eval or exec"` (a call to `eval` or `exec`); `"unpickling"` (`pickle.loads` or `pickle.load`); `"shell=True"` (any call with the keyword `shell=True`); `"certificate checks off"` (any call with `verify=False`); `"SQL built from strings"` (a call to a method named `execute` or `executemany` whose first argument is an f-string with at least one `{}` part, a `+` or `%` expression, or a `.format(...)` call); `"weak hash"` (`hashlib.md5`, `hashlib.sha1`, or `hashlib.new` with the constant `"md5"` or `"sha1"` in any case); `"unsafe yaml"` (`yaml.load` without a `Loader=` keyword whose dotted name contains `Safe`); `"hard-coded secret"` (an assignment of a string constant of at least 8 characters to a name, or an attribute, matching `password`, `secret`, `token`, `api_key` or `apikey`, ignoring case); and `"random used for a secret"` (a call to anything in the `random` module inside a function whose name contains `token`, `password`, `secret` or `key`, ignoring case). Report the line of the call or assignment.

In [ ]:
import ast
import re

def review(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SRC = '''import hashlib, pickle, random, subprocess, yaml, requests\n\nAPI_KEY = \"sk_9f2a77e0c1\"\nDEBUG_PASSWORD = \"short\"\nsettings.secret_key = \"hard-coded-in-code\"\ntimeout = \"a long string but not a secret name\"\n\ndef load(blob, text, name, conn):\n    data = pickle.loads(blob)\n    config = yaml.load(text)\n    safe = yaml.load(text, Loader=yaml.SafeLoader)\n    result = eval(text)\n    subprocess.run(f\"convert {name}\", shell=True)\n    subprocess.run([\"convert\", name])\n    conn.execute(f\"SELECT * FROM t WHERE name = '{name}'\")\n    conn.execute(\"SELECT * FROM t WHERE name = ?\", (name,))\n    conn.execute(\"SELECT * FROM t WHERE id = %s\" % name)\n    conn.executemany(\"INSERT INTO t VALUES ({})\".format(name), [])\n    digest = hashlib.md5(blob).hexdigest()\n    other = hashlib.new(\"SHA1\")\n    good = hashlib.sha256(blob)\n    requests.get(\"https://x.example\", verify=False)\n    return data\n\ndef make_reset_token():\n    return \"\".join(random.choice(\"abcdef\") for _ in range(16))\n\ndef shuffle_cards(cards):\n    random.shuffle(cards)\n'''\n\ndef test_review():\n    \"every rule, and nothing on safe code\"\n    assert review(SRC) == [(3, \"hard-coded secret\"), (5, \"hard-coded secret\"), (9, \"unpickling\"), (10, \"unsafe yaml\"), (12, \"eval or exec\"),\n                           (13, \"shell=True\"), (15, \"SQL built from strings\"), (17, \"SQL built from strings\"), (18, \"SQL built from strings\"),\n                           (19, \"weak hash\"), (20, \"weak hash\"), (22, \"certificate checks off\"), (26, \"random used for a secret\")]\n\ndef test_clean():\n    \"safe code passes\"\n    clean = 'import hashlib\\nPASSWORD_MIN = 12\\nname = \"a fairly long ordinary string\"\\ndef check(conn, x):\\n    conn.execute(\"SELECT 1 WHERE a = ?\", (x,))\\n    conn.execute(f\"SELECT 1\")\\n    return hashlib.sha256(x).hexdigest()\\n'\n    assert review(clean) == []\n    assert review(\"exec('1')\\nexec('2')\") == [(1, \"eval or exec\"), (2, \"eval or exec\")]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: a security review

The starter is a small notes service, `create_app(env, conn, upload_dir, log=print)`, built on miniflask (Week 77) and sqlite3 (Week 75). It works: people can sign up, log in, write notes, search them, view a note as a page, and download files from their own upload folder. It has also failed a security review. Fix it so that it keeps working **and**:

- **Settings**: it refuses to start without a `SECRET_KEY` of at least 32 characters (`RuntimeError("SECRET_KEY must be set (at least 32 characters)")`), and `DEBUG` is on only for `"1"` or `"true"` (any case).
- **Passwords**: stored with the given `hash_password` (salted, slow) and checked with `check_password`; a duplicate email gets `409`.
- **Login**: no SQL injection anywhere (every query uses parameters); one message, `{"error": "wrong email or password"}` with `401`, whether the email or the password was wrong; passwords never appear in the log.
- **Sessions**: the `session` cookie is `USER_ID.SIGNATURE`, the signature being the hex HMAC-SHA256 of the id with the secret key, checked with `hmac.compare_digest`; a missing, unsigned or forged cookie gets `401`. Set it with `httponly=True`, `secure=True` and `samesite="Lax"`.
- **Access**: people see only their **own** notes: someone else's note is `404` (as if it didn't exist), in the JSON and on the page, and searching never shows anyone else's notes, whatever the search text. Search text is matched literally (escape `LIKE` wildcards).
- **Output**: the note page escapes the title and body.
- **Files**: `/files?name=...` serves only files inside the user's own folder (`403` for anything outside, or an empty name; `404` for a missing file).
- **Errors**: an unexpected error returns `500` with only `{"error": "something went wrong", "incident": ID}` (an 8-character hex id), logs the details with that id, and adds `"detail"` only when `DEBUG` is on.

## Forge bonus: an automatic reviewer

Tools such as Bandit read your code as a syntax tree (the `ast` module, Week 69) and flag dangerous patterns. You'll build a small one.

## Weekly quiz

## Week 79 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can find a program's trust boundaries, validate with allow-lists, keep paths inside their folder, fail closed and keep errors and logs from leaking. ([Day 1](w079d1.html))
- I can prevent SQL, command, HTML, log, header and template injection, and refuse internal addresses for user-supplied URLs. ([Day 2](w079d2.html))
- I can keep secrets out of code, logs and URLs, load and validate configuration, scan for leaks and rotate keys. ([Day 3](w079d3.html))
- I can compare versions and specifiers, audit and hash-pin dependencies, trace why a package is installed and spot typosquats. ([Day 4](w079d4.html))
- I know what hashes, MACs, password hashes, authenticated encryption and signatures each promise, and which library function to use for each. ([Day 5](w079d5.html))
- I reviewed and fixed a vulnerable web service, and wrote a small static analyser. (Today)

Tomorrow is a **rest day**. Next week is the **consolidation project**: a complete API built in stages, from design and database models through endpoints, authentication, tests and continuous integration to deployment, bringing together the whole of this phase so far.